# example_pipeline_smoke_test — source scenario generator for `02_pipeline`

Use this notebook after `00_env_config` to create deterministic smoke/demo source tables in the configured `source_lakehouse`. It does **not** run the pipeline guardrails or write unified targets itself. Instead, it prepares small `smoke_` source tables that the real `02_pipeline` template can read to demonstrate the happy path plus schema, DQ, freshness, and load-behavior guardrails.

Safety notes:

- Writes only tables whose names start with `smoke_`.
- Writes to the configured `source_lakehouse` through `write_lakehouse_table(..., CONFIG, ENV, "source", ..., schema=SOURCE_SCHEMA)`.
- Uses overwrite mode so the notebook is safe to rerun.
- Seeds approved active DQ rules only for `smoke_` table names in `METADATA_DQ_RULES`, routed through the configured metadata lakehouse.
- Does not create, replace, or clean non-smoke business tables.
- Keep `example_dq_rule_smoke_test.ipynb` for isolated DQ rule coverage; this notebook is only for end-to-end pipeline scenarios.


## 1. Run `00_env_config`

Run the shared environment notebook first so `CONFIG`, `ENV`, and configured Lakehouse targets are available.


In [ ]:
%run 00_env_config


## 2. Import required functions

The generator uses FabricOps IO helpers only to persist source scenario tables and seed smoke-scoped DQ rule metadata. The real pipeline logic remains in `02_pipeline`.


In [ ]:
import json
from datetime import date, datetime, timedelta

from pyspark.sql.types import DateType, DoubleType, LongType, StringType, StructField, StructType, TimestampType

from fabricops_kit import write_lakehouse_table
from fabricops_kit.config import _current_audit_timestamp


## 3. Define deterministic smoke scenarios

Each source table is prefixed with `smoke_` and represents one scenario that can be selected in `02_pipeline` by changing the source and target table variables in that template.

- `smoke_src_orders_happy` should pass source schema, approved DQ rules, freshness, and load-behavior checks.
- `smoke_src_orders_schema_drift` intentionally omits `order_amount` and adds `promo_code`, so the schema guardrail should fail before the target write.
- `smoke_src_orders_dq_issue` keeps the expected schema but includes null, negative, invalid-status, and duplicate-order records to trigger the approved-rule DQ guardrail.
- `smoke_src_orders_stale` keeps the expected schema but uses old `order_date` / `ingestion_ts` values to trigger the freshness guardrail.
- `smoke_src_orders_reload_a` and `smoke_src_orders_reload_b` support rerunning `02_pipeline` with different row counts and max timestamps to demonstrate append versus overwrite/reload behavior.


In [ ]:
SMOKE_PREFIX = "smoke_"
PIPELINE_DATASET_NAME = "smoke_pipeline_orders"

SOURCE_SCENARIO_TABLES = [
    "smoke_src_orders_happy",
    "smoke_src_orders_schema_drift",
    "smoke_src_orders_dq_issue",
    "smoke_src_orders_stale",
    "smoke_src_orders_reload_a",
    "smoke_src_orders_reload_b",
]

for table_name in SOURCE_SCENARIO_TABLES:
    if not table_name.startswith(SMOKE_PREFIX):
        raise ValueError(f"Refusing to write non-smoke table: {table_name}")

orders_schema = StructType(
    [
        StructField("order_id", LongType(), True),
        StructField("customer_id", LongType(), True),
        StructField("order_date", DateType(), True),
        StructField("ingestion_ts", TimestampType(), True),
        StructField("status", StringType(), True),
        StructField("order_amount", DoubleType(), True),
        StructField("country_code", StringType(), True),
    ]
)

schema_drift_schema = StructType(
    [
        StructField("order_id", LongType(), True),
        StructField("customer_id", LongType(), True),
        StructField("order_date", DateType(), True),
        StructField("ingestion_ts", TimestampType(), True),
        StructField("status", StringType(), True),
        StructField("country_code", StringType(), True),
        StructField("promo_code", StringType(), True),
    ]
)

TODAY = date.today()
FRESH_0 = TODAY
FRESH_1 = TODAY - timedelta(days=1)
STALE = TODAY - timedelta(days=45)


def at_noon(day):
    return datetime(day.year, day.month, day.day, 12, 0, 0)


scenario_frames = {
    "smoke_src_orders_happy": spark.createDataFrame(
        [
            (1001, 501, FRESH_0, at_noon(FRESH_0), "new", 19.99, "US"),
            (1002, 502, FRESH_0, at_noon(FRESH_0), "processing", 125.00, "GB"),
            (1003, 503, FRESH_1, at_noon(FRESH_1), "complete", 42.50, "NL"),
        ],
        orders_schema,
    ),
    "smoke_src_orders_schema_drift": spark.createDataFrame(
        [
            (2001, 601, FRESH_0, at_noon(FRESH_0), "new", "US", "WELCOME10"),
            (2002, 602, FRESH_0, at_noon(FRESH_0), "complete", "CA", ""),
        ],
        schema_drift_schema,
    ),
    "smoke_src_orders_dq_issue": spark.createDataFrame(
        [
            (None, 701, FRESH_0, at_noon(FRESH_0), "new", 25.00, "US"),
            (3002, 702, FRESH_0, at_noon(FRESH_0), "complete", -5.00, "GB"),
            (3003, 703, FRESH_0, at_noon(FRESH_0), "invalid_status", 10.00, "NL"),
            (3003, 704, FRESH_1, at_noon(FRESH_1), "processing", 11.00, "US"),
        ],
        orders_schema,
    ),
    "smoke_src_orders_stale": spark.createDataFrame(
        [
            (4001, 801, STALE, at_noon(STALE), "complete", 75.00, "US"),
            (4002, 802, STALE, at_noon(STALE), "processing", 15.00, "GB"),
        ],
        orders_schema,
    ),
    "smoke_src_orders_reload_a": spark.createDataFrame(
        [
            (5001, 901, FRESH_1, at_noon(FRESH_1), "new", 10.00, "US"),
            (5002, 902, FRESH_1, at_noon(FRESH_1), "complete", 20.00, "GB"),
        ],
        orders_schema,
    ),
    "smoke_src_orders_reload_b": spark.createDataFrame(
        [
            (5001, 901, FRESH_1, at_noon(FRESH_1), "new", 10.00, "US"),
            (5002, 902, FRESH_1, at_noon(FRESH_1), "complete", 20.00, "GB"),
            (5003, 903, FRESH_0, at_noon(FRESH_0), "processing", 30.00, "NL"),
            (5004, 904, FRESH_0, at_noon(FRESH_0), "complete", 40.00, "CA"),
        ],
        orders_schema,
    ),
}


## 4. Create or replace source scenario tables

This cell physically writes each scenario DataFrame to the configured `source_lakehouse`. It uses overwrite mode and refuses to write names that do not start with `smoke_`, making the notebook safe to rerun without touching non-smoke assets.


In [ ]:
write_results = []
for table_name, dataframe in scenario_frames.items():
    if not table_name.startswith(SMOKE_PREFIX):
        raise ValueError(f"Refusing to write non-smoke table: {table_name}")
    write_lakehouse_table(
        dataframe,
        CONFIG,
        ENV,
        "source",
        table_name,
        schema=SOURCE_SCHEMA,
        mode="overwrite",
        options={"overwriteSchema": "true"},
    )
    write_results.append({"source_table": table_name, "row_count": dataframe.count(), "write_mode": "overwrite"})

write_results_df = spark.createDataFrame(write_results)
display(write_results_df.orderBy("source_table"))


## 5. Seed smoke-scoped approved DQ rules for `02_pipeline`

`02_pipeline` enforces approved active rules from `METADATA_DQ_RULES` when `dq_preset` is `approved_rules`. This cell appends public-safe DQ rule metadata for the smoke order tables only, using stable `rule_key` values so reruns supersede earlier smoke rule rows through the normal append-only metadata versioning path.

The schema-drift table intentionally has no DQ rules so the schema guardrail remains the visible failure.


In [ ]:
now_utc = _current_audit_timestamp(config=CONFIG, drop_microseconds=False)

DQ_RULE_TABLES = [
    "smoke_src_orders_happy",
    "smoke_src_orders_dq_issue",
    "smoke_src_orders_stale",
    "smoke_src_orders_reload_a",
    "smoke_src_orders_reload_b",
]

DQ_RULE_DEFINITIONS = [
    {"rule_type": "not_null", "columns": ["order_id"], "severity": "error", "description": "Smoke orders require order_id."},
    {"rule_type": "unique", "columns": ["order_id"], "severity": "error", "description": "Smoke orders should not duplicate order_id."},
    {"rule_type": "greater_than_or_equal", "columns": ["order_amount"], "value": 0, "severity": "error", "description": "Smoke orders require non-negative order_amount."},
    {"rule_type": "accepted_values", "columns": ["status"], "allowed_values": ["new", "processing", "complete", "cancelled"], "severity": "error", "description": "Smoke orders require a known status."},
]


def metadata_key(table_name, column_name=""):
    return "||".join([str(ENV), PIPELINE_DATASET_NAME, table_name, column_name])


def build_smoke_rule_rows(table_name):
    rows = []
    for rule in DQ_RULE_DEFINITIONS:
        rule_type = rule["rule_type"]
        first_column = list(rule.get("columns", []))[0]
        rule_id = f"smoke_pipeline_orders__{table_name}__{rule_type}"
        parameters = {key: value for key, value in rule.items() if key not in {"rule_type", "severity", "description"}}
        rows.append(
            {
                "rule_key": metadata_key(table_name, rule_id),
                "rule_id": rule_id,
                "metadata_column_key": metadata_key(table_name, first_column),
                "metadata_table_key": metadata_key(table_name),
                "environment_name": ENV,
                "dataset_name": PIPELINE_DATASET_NAME,
                "table_name": table_name,
                "column_name": first_column,
                "rule_type": rule_type,
                "rule_parameters_json": json.dumps(parameters, sort_keys=True),
                "severity": rule["severity"],
                "description": rule["description"],
                "is_active": True,
                "review_status": "approved",
                "approved_by": "example_pipeline_smoke_test",
                "approved_at": now_utc,
                "ai_suggestion_json": "{}",
                "action_type": "created",
                "_committed_at": now_utc,
                "_committed_by": "example_pipeline_smoke_test",
                "_workspace_name": "",
                "_notebook_name": "example_pipeline_smoke_test",
                "_metadata_lakehouse_name": CONFIG.path_config.paths[ENV]["metadata"].name,
                "_activity_id": "example_pipeline_smoke_test",
            }
        )
    return rows

seeded_rule_rows = []
for source_table in DQ_RULE_TABLES:
    if not source_table.startswith(SMOKE_PREFIX):
        raise ValueError(f"Refusing to seed DQ rules for non-smoke table: {source_table}")
    seeded_rule_rows.extend(build_smoke_rule_rows(source_table))

seeded_rule_df = spark.createDataFrame(seeded_rule_rows)
write_lakehouse_table(seeded_rule_df, CONFIG, ENV, "metadata", "METADATA_DQ_RULES", schema=METADATA_SCHEMA, mode="append")
display(seeded_rule_df.select("dataset_name", "table_name", "rule_type", "severity", "review_status").orderBy("table_name", "rule_type"))


## 6. Use these tables in `02_pipeline`

In `02_pipeline`, keep the orchestration cells unchanged and update only the source/target configuration variables near the load-source section. For example:

```python
PIPELINE_SOURCE_TABLE_NAME = "smoke_src_orders_happy"
PIPELINE_TARGET_TABLE_NAME = "smoke_unified_orders_happy"
PIPELINE_TARGET_WRITE_MODE = "overwrite"
PIPELINE_LOAD_BEHAVIOR = "append"
# PIPELINE_DATASET_NAME is derived automatically for smoke_ sources; override only when needed.
```

For the reload demo, run once with `smoke_src_orders_reload_a`, then rerun with `smoke_src_orders_reload_b`. Compare `PIPELINE_TARGET_WRITE_MODE = "append"` versus `"overwrite"` and `PIPELINE_LOAD_BEHAVIOR = "append"` versus `"overwrite"` to show how the real template handles row-count and watermark changes.

The pipeline guardrail summary tables should show whether each guardrail passed, warned, failed, or blocked the target write. Catalogue, DQ, lineage, and run-summary evidence continues to use the metadata lakehouse configured by `00_env_config`.


## 7. Scenario catalogue

The catalogue below is the handoff checklist for selecting each smoke source table in `02_pipeline`. All intended target table names are also prefixed with `smoke_`.


In [ ]:
scenario_catalogue_rows = [
    {
        "scenario_name": "happy_path",
        "source_table": "smoke_src_orders_happy",
        "intended_target_table": "smoke_unified_orders_happy",
        "guardrail_demonstrated": "schema + DQ + freshness + load behavior pass",
        "expected_02_pipeline_result": "passes source and target guardrails, writes smoke target in unified_lakehouse, records metadata evidence",
        "demo_notes": "Use overwrite for a clean first run; approved smoke DQ rules should pass.",
    },
    {
        "scenario_name": "schema_drift",
        "source_table": "smoke_src_orders_schema_drift",
        "intended_target_table": "smoke_unified_orders_schema_drift",
        "guardrail_demonstrated": "schema guardrail failure",
        "expected_02_pipeline_result": "source schema guardrail fails and blocks the target write",
        "demo_notes": "Table omits order_amount and adds promo_code; no DQ rules are seeded so schema drift is the visible guardrail.",
    },
    {
        "scenario_name": "dq_issue",
        "source_table": "smoke_src_orders_dq_issue",
        "intended_target_table": "smoke_unified_orders_dq_issue",
        "guardrail_demonstrated": "DQ guardrail failure",
        "expected_02_pipeline_result": "approved DQ rules fail and block the target write",
        "demo_notes": "Includes null order_id, duplicate order_id, negative order_amount, and invalid status.",
    },
    {
        "scenario_name": "stale_source",
        "source_table": "smoke_src_orders_stale",
        "intended_target_table": "smoke_unified_orders_stale",
        "guardrail_demonstrated": "freshness guardrail failure",
        "expected_02_pipeline_result": "freshness guardrail fails and blocks the target write",
        "demo_notes": "Valid schema and DQ values, but order_date and ingestion_ts are intentionally old.",
    },
    {
        "scenario_name": "reload_a",
        "source_table": "smoke_src_orders_reload_a",
        "intended_target_table": "smoke_unified_orders_reload_demo",
        "guardrail_demonstrated": "load behavior baseline / first run",
        "expected_02_pipeline_result": "establishes the first smoke profile and target state for reload comparison",
        "demo_notes": "Run this first for the reload demo.",
    },
    {
        "scenario_name": "reload_b",
        "source_table": "smoke_src_orders_reload_b",
        "intended_target_table": "smoke_unified_orders_reload_demo",
        "guardrail_demonstrated": "load behavior row-count and max-timestamp change",
        "expected_02_pipeline_result": "supports comparing append versus overwrite/reload behavior in the real pipeline template",
        "demo_notes": "Run this second against the same target as reload_a, changing write/load behavior variables as needed.",
    },
]

scenario_catalogue_df = spark.createDataFrame(scenario_catalogue_rows).select(
    "scenario_name",
    "source_table",
    "intended_target_table",
    "guardrail_demonstrated",
    "expected_02_pipeline_result",
    "demo_notes",
)
display(scenario_catalogue_df.orderBy("scenario_name"))
